# SUMMARY — all research questions, all datasets, all seeds

Reads every run folder produced by the 21 notebooks and reports RQ1–RQ4 across three
datasets and three seeds. **Nothing is trained, nothing is recomputed.** This notebook opens
other runs read-only and writes only into its own folder.

## The reporting rule, fixed before any number is read

Nine RQ2/RQ3 runs invite reporting whichever seed looks best. That is foreclosed here:

1. **Seed 20260202 (seed_index 0) is the pre-registered primary.** RQ2 and RQ3 are reported
   from it. That was the plan before any of this ran and it does not change now.
2. **Seeds 20260203 and 20260204 are robustness evidence, not alternatives.** They answer
   "does the primary conclusion survive a reseed?", never "which seed do we prefer?"
3. **Disagreement across seeds is reported, not resolved.** A sign that flips or an MMDE
   verdict that changes between seeds is itself the finding.
4. **Every dataset is reported, including the ones that do not cooperate.**

A conclusion that holds on the primary seed but not the other two is a fragile conclusion,
and this notebook is built to say so out loud.

## RUN IDENTITY

| | |
|---|---|
| **Run ID / folder** | `SUMMARY__all` |
| **Reads (read-only)** | all 9 RQ1, 9 RQ2RQ3 and 3 RQ4 run folders |
| **Writes** | `/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/SUMMARY__all/` only |
| **Hardware** | CPU, seconds |

Run this last. It tolerates missing runs and reports what is absent rather than quietly omitting it.

In [1]:
# --- locate pcdt_p2.py and p2_runtime.py -----------------------------------------
import os, sys, json, glob
_here = os.path.abspath("")
for _c in (_here, os.path.dirname(_here), os.path.join(_here, "..")):
    if os.path.exists(os.path.join(_c, "pcdt_p2.py")):
        sys.path.insert(0, os.path.abspath(_c)); break
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg") if os.environ.get("P2_HEADLESS") else None
import matplotlib.pyplot as plt
import pcdt_p2 as P
import p2_runtime as R
pd.set_option("display.width", 220)
print("pcdt_p2   :", P.__file__)
print("p2_runtime:", R.__file__)
# --- AUDIT GUARD (CORRECTIONS.md) -------------------------------------------------
# Refuse to run against the pre-audit library. Without this a stale pcdt_p2.py on the
# path silently reinstates the DKVMN memory-ordering defect and every table below is
# quietly wrong again.
assert hasattr(P.Config(), "dkvmn_legacy_lag"), (
    "the pcdt_p2.py on this path is the PRE-AUDIT copy: it has no dkvmn_legacy_lag "
    "field, so DKVMN still reads its value memory before writing the current "
    "interaction. Replace pcdt_p2.py and p2_runtime.py with the corrected versions.")
assert hasattr(P, "check_alignment"), "corrected pcdt_p2.py is missing check_alignment"

pcdt_p2   : /home/hamed/Downloads/Research/P02/pcdt_p2.py
p2_runtime: /home/hamed/Downloads/Research/P02/p2_runtime.py


In [2]:
# ============================================================================
# CONFIGURATION — GENERATED. Nothing to edit.
# This notebook is a CONSUMER: it opens every producer folder read-only and writes
# only into its own. finalize() fails the run if any foreign file changed.
# ============================================================================
RUN_ID   = "SUMMARY__all"
OUT_ROOT = "/home/hamed/Downloads/Research/P02/PCDT_P2_runs"

DATASETS = ["assistments_2021", "ednet_flat", "junyi"]
SEEDS    = {0: 20260202, 1: 20260203, 2: 20260204}
PRIMARY_SEED_INDEX = 0            # pre-registered; not re-chosen here

# PCDT-noPsi and Control are PCDT variants, not baselines. The strongest COMPARATOR is
# the best tuned KT model, which is what RQ2 is judged against.
BASELINES = ["DKT", "DKT-F", "SAKT", "AKT", "DKVMN"]
MMDE = P.Config().min_meaningful_auc_delta

RQ1_ID    = lambda ds, si: f"RQ1__{ds}__seed{si}"
RQ2RQ3_ID = lambda ds, si: f"RQ2RQ3__{ds}__seed{si}"
RQ4_ID    = lambda ds: f"RQ4__{ds}"

CTX = R.RunContext(RUN_ID, OUT_ROOT,
                   inputs_expected=["all RQ1, RQ2RQ3 and RQ4 run folders"],
                   description="Cross-run summary — RQ1-RQ4, 3 datasets, 3 seeds")
DIRS = CTX.dirs
print(f"\nminimum meaningful AUC difference (pre-registered): {MMDE}")
print(f"primary seed: {SEEDS[PRIMARY_SEED_INDEX]}")

RUN SUMMARY__all
Cross-run summary — RQ1-RQ4, 3 datasets, 3 seeds
started   2026-08-18 17:03:20   pid 600307
writes to /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/SUMMARY__all
reads     all RQ1, RQ2RQ3 and RQ4 run folders

minimum meaningful AUC difference (pre-registered): 0.01
primary seed: 20260202


## 1. Inventory — which runs exist, and did each stay in its lane

Every run writes a `manifest.json` recording its status and an isolation check. A run whose
`isolation_ok` is false wrote into another notebook's folder and must not be trusted.

In [3]:
rows = []
for kind, ids in (("RQ1", [(RQ1_ID(d, s), d, SEEDS[s]) for d in DATASETS for s in SEEDS]),
                  ("RQ2RQ3", [(RQ2RQ3_ID(d, s), d, SEEDS[s]) for d in DATASETS for s in SEEDS]),
                  ("RQ4", [(RQ4_ID(d), d, None) for d in DATASETS])):
    for rid, ds, seed in ids:
        man = R.load_manifest(CTX.runs_root, rid)
        rows.append(dict(kind=kind, run_id=rid, dataset=ds, seed=seed,
                         present=man is not None,
                         status=(man or {}).get("status", "-"),
                         files=(man or {}).get("n_outputs", 0),
                         minutes=(man or {}).get("runtime_minutes", np.nan),
                         isolation_ok=(man or {}).get("isolation_ok", None),
                         fixture=(man or {}).get("fixture", None)))
INVENTORY = pd.DataFrame(rows)
print(INVENTORY.to_string(index=False))

absent = INVENTORY[~INVENTORY.present]
if len(absent):
    print(f"\n*** {len(absent)} run(s) not yet executed:")
    print("    " + ", ".join(absent.run_id))
    print("    Everything below is computed only from the runs that DO exist.")

bad = INVENTORY[INVENTORY.isolation_ok == False]
if len(bad):
    print(f"\n*** ISOLATION FAILURE in {len(bad)} run(s): {list(bad.run_id)}")
    print("    Those runs modified files outside their own folder. Re-run them before "
          "reporting anything derived from them.")

fx = INVENTORY[INVENTORY.fixture == True]
if len(fx):
    print(f"\n*** {len(fx)} run(s) executed in FIXTURE mode — synthetic data, NOT results: "
          f"{list(fx.run_id)}")

  kind                          run_id          dataset       seed  present   status  files  minutes  isolation_ok  fixture
   RQ1    RQ1__assistments_2021__seed0 assistments_2021 20260202.0     True complete     12     0.54          True    False
   RQ1    RQ1__assistments_2021__seed1 assistments_2021 20260203.0     True complete     12     0.55          True    False
   RQ1    RQ1__assistments_2021__seed2 assistments_2021 20260204.0     True complete     12     0.52          True    False
   RQ1          RQ1__ednet_flat__seed0       ednet_flat 20260202.0     True complete     12     0.57          True    False
   RQ1          RQ1__ednet_flat__seed1       ednet_flat 20260203.0     True complete     12     0.52          True    False
   RQ1          RQ1__ednet_flat__seed2       ednet_flat 20260204.0     True complete     12     0.53          True    False
   RQ1               RQ1__junyi__seed0            junyi 20260202.0     True complete     12     1.23          True    False
   RQ1  

## 2. RQ1 — does the construct-validity verdict hold across datasets and seeds?

The audit trains nothing, but the seed still draws the 20,000-learner subsample, so each
seed audits a different sample. A verdict that moves between seeds is sample-dependent and
cannot be reported as a property of the dataset.

This cross-dataset comparison lives here rather than in the RQ1 notebooks: each of those
sees one dataset at one seed and must not imply otherwise.

In [4]:
rows = []
for ds in DATASETS:
    for si in SEEDS:
        base = CTX.sibling(RQ1_ID(ds, si))
        if base is None:
            continue
        f = CTX.read_input(os.path.join(base, "tables", f"table1_audit_{ds}.csv"),
                           label=f"{RQ1_ID(ds, si)}/table1", required=False)
        if not f:
            continue
        t = pd.read_csv(f)
        for r in t.itertuples():
            rows.append(dict(dataset=ds, seed=SEEDS[si], dimension=r.dimension,
                             convergent=getattr(r, "convergent", np.nan), verdict=r.verdict))
RQ1 = pd.DataFrame(rows)

if RQ1.empty:
    print("no RQ1 runs found — execute the nine 01_RQ1__* notebooks first")
else:
    piv = RQ1.pivot_table(index="dimension", columns=["dataset", "seed"],
                          values="verdict", aggfunc="first")
    print("Verdict by dimension x dataset x seed\n")
    print(piv.to_string())

    unstable_seed = []
    for ds in DATASETS:
        sub = RQ1[RQ1.dataset == ds]
        for dim, g in sub.groupby("dimension"):
            if g.verdict.nunique() > 1:
                unstable_seed.append(dict(dataset=ds, dimension=dim,
                                          verdicts="; ".join(f"{s}:{v}" for s, v
                                                             in zip(g.seed, g.verdict))))
    print("\nA. Verdicts that CHANGE ACROSS SEEDS within a dataset:")
    print("   none — the audit is seed-stable" if not unstable_seed
          else pd.DataFrame(unstable_seed).to_string(index=False))

    prim = RQ1[RQ1.seed == SEEDS[PRIMARY_SEED_INDEX]]
    if not prim.empty:
        m = prim.pivot_table(index="dimension", columns="dataset", values="verdict",
                             aggfunc="first")
        agree = m.apply(lambda r: r.dropna().nunique() == 1, axis=1)
        print("\nB. On the PRIMARY seed, verdict by dataset:\n")
        print(m.to_string())
        print(f"\n   identical on every dataset : {list(m.index[agree])}")
        print(f"   varies by dataset          : {list(m.index[~agree])}")

  read input     RQ1__assistments_2021__seed0/table1
  read input     RQ1__assistments_2021__seed1/table1
  read input     RQ1__assistments_2021__seed2/table1
  read input     RQ1__ednet_flat__seed0/table1
  read input     RQ1__ednet_flat__seed1/table1
  read input     RQ1__ednet_flat__seed2/table1
  read input     RQ1__junyi__seed0/table1
  read input     RQ1__junyi__seed1/table1
  read input     RQ1__junyi__seed2/table1
Verdict by dimension x dataset x seed

dataset                 assistments_2021                   ednet_flat                      junyi                  
seed                            20260202 20260203 20260204   20260202 20260203 20260204 20260202 20260203 20260204
dimension                                                                                                         
B_t (behaviour)                   reject   reject   reject     reject   reject   reject   reject   reject   reject
C_t (cognition)                   retain   retain   retain     retain   ret

## 3. RQ2 — within-session prediction

PCDT against the strongest tuned KT baseline, judged against the pre-registered minimum
meaningful difference. **The MMDE verdict is the result, not the sign of the delta** — a
positive delta below the MMDE is equivalence, not a win.

In [5]:
rows = []
for ds in DATASETS:
    for si in SEEDS:
        base = CTX.sibling(RQ2RQ3_ID(ds, si))
        if base is None:
            continue
        f = CTX.read_input(os.path.join(base, "tables", "table2_predictive.csv"),
                           label=f"{RQ2RQ3_ID(ds, si)}/table2", required=False)
        if not f:
            continue
        t = pd.read_csv(f).set_index("model")["auc"]
        avail = [b for b in BASELINES if b in t.index]
        if "PCDT" not in t.index or not avail:
            continue
        best = max(avail, key=lambda b: t[b])
        delta = float(t["PCDT"] - t[best])
        rows.append(dict(dataset=ds, seed=SEEDS[si], primary=(si == PRIMARY_SEED_INDEX),
                         pcdt_auc=float(t["PCDT"]), strongest=best,
                         baseline_auc=float(t[best]), delta=delta,
                         verdict=("PCDT higher" if delta >= MMDE else
                                  "PCDT lower" if delta <= -MMDE else
                                  "equivalent (below MMDE)")))
RQ2 = pd.DataFrame(rows)

if RQ2.empty:
    print("no RQ2 results found")
else:
    print(RQ2.round(4).to_string(index=False))
    print("\n" + "=" * 78)
    print("PRIMARY SEED — this is what RQ2 is reported on")
    print("=" * 78)
    print(RQ2[RQ2.primary].round(4)[["dataset", "pcdt_auc", "strongest",
                                     "baseline_auc", "delta", "verdict"]]
          .to_string(index=False))
    print("\nStability across seeds:")
    for ds, g in RQ2.groupby("dataset"):
        vs = sorted(set(g.verdict))
        flag = "STABLE" if len(vs) == 1 else "*** UNSTABLE ***"
        print(f"  {ds:18s} delta {g.delta.mean():+.4f} +/- {g.delta.std(ddof=0):.4f} "
              f"over {len(g)} seed(s)   {flag}")
        if len(vs) > 1:
            print(f"      verdicts: {dict(zip(g.seed, g.verdict))}")
        if g.strongest.nunique() > 1:
            print(f"      note: the strongest baseline is not the same model in every "
                  f"seed: {dict(zip(g.seed, g.strongest))}")

  read input     RQ2RQ3__assistments_2021__seed0/table2
  read input     RQ2RQ3__assistments_2021__seed1/table2
  read input     RQ2RQ3__assistments_2021__seed2/table2
  read input     RQ2RQ3__ednet_flat__seed0/table2
  read input     RQ2RQ3__ednet_flat__seed1/table2
  read input     RQ2RQ3__ednet_flat__seed2/table2
  read input     RQ2RQ3__junyi__seed0/table2
  read input     RQ2RQ3__junyi__seed1/table2
  read input     RQ2RQ3__junyi__seed2/table2
         dataset     seed  primary  pcdt_auc strongest  baseline_auc   delta                 verdict
assistments_2021 20260202     True    0.7506     DKT-F        0.7493  0.0013 equivalent (below MMDE)
assistments_2021 20260203    False    0.7505     DKT-F        0.7494  0.0011 equivalent (below MMDE)
assistments_2021 20260204    False    0.7493     DKT-F        0.7504 -0.0011 equivalent (below MMDE)
      ednet_flat 20260202     True    0.7117     DKVMN        0.7111  0.0006 equivalent (below MMDE)
      ednet_flat 20260203    False    0.71

## 4. RQ3 — retention across inter-session gaps

Does the persistence advantage over each comparator keep its sign *and* its significance
when the seed changes? Rows the producer marked inadequate are excluded — on ASSISTments and
EdNet that removes d28.

In [6]:
rows = []
for ds in DATASETS:
    for si in SEEDS:
        base = CTX.sibling(RQ2RQ3_ID(ds, si))
        if base is None:
            continue
        f = CTX.read_input(os.path.join(base, "tables", "table5_persistence.csv"),
                           label=f"{RQ2RQ3_ID(ds, si)}/table5", required=False)
        if not f:
            continue
        t = pd.read_csv(f)
        if "adequate" in t.columns:
            t = t[t.adequate.astype(str).str.lower().isin(["true", "1", "yes"])]
        for r in t.itertuples():
            lo, hi = float(r.ci_lo), float(r.ci_hi)
            # A NaN interval means the bootstrap could not be formed. That is missing
            # evidence, NOT evidence of no effect, and must not be reported as "null".
            # AUDIT FIX (CORRECTIONS.md, Finding C): the label was the literal string
            # "null", which pandas.read_csv converts back to NaN by default. A genuine
            # "CI includes zero" result was therefore indistinguishable on re-read from a
            # bootstrap that could not be formed — the exact conflation the comment above
            # exists to prevent, in the opposite direction.
            sig = ("n/a (no CI)" if (np.isnan(lo) or np.isnan(hi))
                   else "positive" if lo > 0 else "negative" if hi < 0
                   else "inconclusive (CI spans zero)")
            rows.append(dict(dataset=ds, seed=SEEDS[si], primary=(si == PRIMARY_SEED_INDEX),
                             horizon=r.horizon, comparator=r.comparator,
                             advantage=float(r.retention_advantage),
                             ci_lo=lo, ci_hi=hi, sig=sig))
RQ3 = pd.DataFrame(rows)

if RQ3.empty:
    print("no RQ3 results found")
else:
    print("PRIMARY SEED — adequate horizons only\n")
    print(RQ3[RQ3.primary].round(4)[["dataset", "horizon", "comparator", "advantage",
                                     "ci_lo", "ci_hi", "sig"]].to_string(index=False))
    print("\n" + "=" * 78)
    print("CROSS-SEED STABILITY of each (dataset, horizon, comparator) cell")
    print("=" * 78)
    flips = []
    for key, g in RQ3.groupby(["dataset", "horizon", "comparator"]):
        g = g[g.sig != "n/a (no CI)"]        # a missing CI cannot disagree with anything
        if len(g) < 2 or g.sig.nunique() == 1:
            continue
        flips.append(dict(dataset=key[0], horizon=key[1], comparator=key[2],
                          verdicts="; ".join(f"{s}:{v}" for s, v in zip(g.seed, g.sig)),
                          spread=float(g.advantage.max() - g.advantage.min())))
    if flips:
        print(pd.DataFrame(flips).round(4).to_string(index=False))
        print("\n*** These cells do not replicate across seeds. Report them as unstable, "
              "not as the primary seed's verdict. ***")
    else:
        print("every adequate cell keeps the same significance verdict across all seeds")

  read input     RQ2RQ3__assistments_2021__seed0/table5
  read input     RQ2RQ3__assistments_2021__seed1/table5
  read input     RQ2RQ3__assistments_2021__seed2/table5
  read input     RQ2RQ3__ednet_flat__seed0/table5
  read input     RQ2RQ3__ednet_flat__seed1/table5
  read input     RQ2RQ3__ednet_flat__seed2/table5
  read input     RQ2RQ3__junyi__seed0/table5
  read input     RQ2RQ3__junyi__seed1/table5
  read input     RQ2RQ3__junyi__seed2/table5
PRIMARY SEED — adequate horizons only

         dataset horizon comparator  advantage   ci_lo   ci_hi                          sig
assistments_2021      d1    Control     0.0195  0.0168  0.0222                     positive
assistments_2021      d7    Control     0.0211  0.0168  0.0252                     positive
assistments_2021      d1      DKT-F    -0.0087 -0.0125 -0.0051                     negative
assistments_2021      d7      DKT-F    -0.0092 -0.0155 -0.0027                     negative
      ednet_flat      d1    Control     0.0061  

## 5. RQ4 — ranking divergence, all three datasets

Each dataset's verdict comes from its own confirmatory seed (20260204) against the rule
fixed in that dataset's RQ4 notebook before any result was read.

**All three datasets appear here regardless of outcome.** Running three and reporting only
the ones that confirm would be selection on the result — the very practice RQ4 criticises in
within-session benchmarking.

In [7]:
rows = []
for ds in DATASETS:
    base = CTX.sibling(RQ4_ID(ds))
    if base is None:
        rows.append(dict(dataset=ds, status="NOT RUN", comparator="-",
                         confirmed="-", horizons_used="-", failing_leg="-"))
        continue
    man = R.load_manifest(CTX.runs_root, RQ4_ID(ds)) or {}
    f = CTX.read_input(os.path.join(base, "artifacts", "verdict_rq4.json"),
                       label=f"{RQ4_ID(ds)}/verdict", required=False)
    # AUDIT FIX (CORRECTIONS.md, Finding B): verdict_rq4.json stores the verdict one
    # level down, under "verdict". Reading the top level returned None for every field, so
    # this table printed "-" for all three datasets and the dataset-dependence warning
    # below could never fire — on results that are in fact 2-of-3 confirmed.
    rec = json.load(open(f)) if f else {}
    v = rec.get("verdict", rec) or man.get("verdict", {}) or {}
    pc = v.get("primary_confirmed")
    # AUDIT FIX: "adequate_horizons" was never written by the archived notebook 03, so this
    # field stayed blank even with the nesting corrected. Fall back to the confirmatory
    # seed's table9 — the file the rule itself was evaluated on.
    horizons = v.get("adequate_horizons") or []
    if not horizons:
        b23 = CTX.sibling(RQ2RQ3_ID(ds, max(SEEDS)))
        if b23:
            t9 = os.path.join(b23, "tables", "table9_postgap_auc_by_model.csv")
            if os.path.exists(t9):
                _t = pd.read_csv(t9)
                horizons = [h for h in P.REAL_HORIZONS
                            if f"auc_{h}" in _t.columns and _t[f"auc_{h}"].notna().any()]
    _sens = (v.get("selection_sensitivity") or {})
    rows.append(dict(dataset=ds, status=man.get("status", "run"),
                     comparator=v.get("comparator", "-"),
                     confirmed=("-" if pc is None else
                                ("CONFIRMED" if pc else "NOT CONFIRMED")),
                     secondary=("-" if v.get("secondary_confirmed") is None
                                else ("passed" if v.get("secondary_confirmed")
                                      else "not passed")),
                     selection_sensitive=("-" if not _sens else
                                          str(bool(_sens.get("divergence") != pc))),
                     horizons_used=", ".join(horizons) or "-",
                     failing_leg=v.get("failing_leg", "-")))
RQ4 = pd.DataFrame(rows)
print(RQ4.to_string(index=False))

done = RQ4[RQ4.confirmed.isin(["CONFIRMED", "NOT CONFIRMED"])]
if len(done):
    n_ok = int((done.confirmed == "CONFIRMED").sum())
    print(f"\nDivergence confirmed on {n_ok} of {len(done)} dataset(s) tested.")
    if 0 < n_ok < len(done):
        print("*** The effect is DATASET-DEPENDENT. A claim about the field's default "
              "evaluation protocol cannot rest on the subset where it replicated — report "
              "the datasets where it did not. ***")
    elif n_ok == len(done) == 3:
        print("Replicates on all three datasets — the strongest form of this claim these "
              "data support.")
print("\nd28 is below the adequacy floor on ASSISTments and EdNet, so leg 2 of the rule was "
      "tested on d1 and d7 only there. State the horizons used alongside each verdict.")

  read input     RQ4__assistments_2021/verdict
  read input     RQ4__ednet_flat/verdict
  read input     RQ4__junyi/verdict
         dataset   status comparator     confirmed  secondary selection_sensitive horizons_used                                            failing_leg
assistments_2021 complete PCDT-noPsi NOT CONFIRMED not passed               False        d1, d7 leg1: comparator ranks inside the top 3 within session
      ednet_flat complete      DKVMN NOT CONFIRMED not passed               False        d1, d7 leg1: comparator ranks inside the top 3 within session
           junyi complete      DKVMN NOT CONFIRMED     passed               False   d1, d7, d28 leg1: comparator ranks inside the top 3 within session

Divergence confirmed on 0 of 3 dataset(s) tested.

d28 is below the adequacy floor on ASSISTments and EdNet, so leg 2 of the rule was tested on d1 and d7 only there. State the horizons used alongside each verdict.


## 6. Write the summary tables and close the run

In [8]:
tabs = {}
for name, t in [("summary_inventory", INVENTORY), ("summary_rq1_verdicts", RQ1),
                ("summary_rq2_cross_seed", RQ2), ("summary_rq3_cross_seed", RQ3),
                ("summary_rq4_by_dataset", RQ4)]:
    if isinstance(t, pd.DataFrame) and not t.empty:
        tabs[name] = t
P.save_tables(tabs, DIRS["tab"])
print(f"wrote {len(tabs)} table(s) -> {DIRS['tab']}")

n_missing = int((~INVENTORY.present).sum())
CTX.finalize(
    status=("complete" if n_missing == 0 else f"partial — {n_missing} run(s) not executed"),
    notes=("; ".join(INVENTORY.loc[~INVENTORY.present, "run_id"]) if n_missing else ""),
    extra=dict(reporting_rule=("RQ2/RQ3 reported on the pre-registered primary seed "
                               "20260202; other seeds are robustness evidence; "
                               "cross-seed disagreement is reported, not resolved"),
               primary_seed=SEEDS[PRIMARY_SEED_INDEX],
               runs_expected=int(len(INVENTORY)),
               runs_present=int(INVENTORY.present.sum())))

tables -> /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/SUMMARY__all/tables: ['summary_inventory', 'summary_rq1_verdicts', 'summary_rq2_cross_seed', 'summary_rq3_cross_seed', 'summary_rq4_by_dataset']
wrote 5 table(s) -> /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/SUMMARY__all/tables

RUN SUMMARY__all — complete
  runtime        0.0 min
  files written  11
  inputs read    30 (0 missing)
  isolation      OK — no finished run was modified
  manifest       /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/SUMMARY__all/manifest.json


{'run_id': 'SUMMARY__all',
 'description': 'Cross-run summary — RQ1-RQ4, 3 datasets, 3 seeds',
 'status': 'complete',
 'notes': '',
 'fixture': False,
 'started': '2026-08-18 17:03:20',
 'finished': '2026-08-18 17:03:21',
 'runtime_minutes': 0.0,
 'output_dir': '/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/SUMMARY__all',
 'environment': {'python': '3.13.9',
  'platform': 'Linux-7.0.0-28-generic-x86_64-with-glibc2.39',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'torch': '2.12.0.dev20260408+cu128',
  'sklearn': '1.7.2',
  'scipy': '1.18.0',
  'matplotlib': '3.10.6',
  'cuda': True,
  'gpu': 'NVIDIA GeForce RTX 5080'},
 'config': None,
 'inputs': [{'label': 'RQ1__assistments_2021__seed0/table1',
   'path': '/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ1__assistments_2021__seed0/tables/table1_audit_assistments_2021.csv',
   'present': True,
   'sha256': '942badceb68e2b557261cf2b831141a24dfeaa3267a43566136c401ec3801e97',
   'bytes': 1602,
   'mtime': '2026-08-17 20:00:16'}